# PHÂN TÍCH ĐÁNH GIÁ KHÁCH HÀNG — SO SÁNH 4 MÔ HÌNH
**Ba ML (Naive Bayes, Logistic Regression, Random Forest) vs PyTorch MLP**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import matplotlib
matplotlib.use('Agg')

MODEL_DIR = os.path.join('..', 'models')
data = np.load(os.path.join(MODEL_DIR, 'tfidf_data.npz'))
X_test, y_test = data['X_test'], data['y_test']

models = ['Multinomial Naive Bayes', 'Logistic Regression', 'Random Forest']
results = []

for m in models:
    model = joblib.load(os.path.join(MODEL_DIR, f"cb_{m.lower().replace(' ', '_')}.pkl"))
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    results.append({
        'Model': m,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds, zero_division=0),
        'Recall': recall_score(y_test, preds, zero_division=0),
        'F1-Score': f1_score(y_test, preds, zero_division=0),
        'AUC-ROC': roc_auc_score(y_test, probs)
    })

# Dữ liệu DL
dl_data = np.load(os.path.join(MODEL_DIR, 'cb_dl_preds.npz'))
dl_preds = dl_data['preds']
dl_probs = dl_data['probs']
results.append({
    'Model': 'Deep Learning (PyTorch MLP)',
    'Accuracy': accuracy_score(y_test, dl_preds),
    'Precision': precision_score(y_test, dl_preds, zero_division=0),
    'Recall': recall_score(y_test, dl_preds, zero_division=0),
    'F1-Score': f1_score(y_test, dl_preds, zero_division=0),
    'AUC-ROC': roc_auc_score(y_test, dl_probs)
})

df_comp = pd.DataFrame(results)
pd.set_option('display.float_format', '{:.4f}'.format)
print(df_comp)

fig, ax = plt.subplots(figsize=(10,6))
df_comp.set_index('Model')[['F1-Score', 'AUC-ROC']].plot(kind='bar', ax=ax, width=0.7, rot=15)
ax.set_ylim(0, 1.1)
ax.set_title('So Sánh F1-Score & AUC-ROC giữa 4 mô hình')
plt.tight_layout(); plt.show()


## Kết luận
Với dữ liệu Text (TF-IDF), **Logistic Regression** và **Deep Learning MLP** thường cho kết quả bám sát nhau và cao nhất. Naive Bayes chạy rất nhanh nhưng Recall đôi khi thấp. Tree-based model (Random Forest) hoạt động không quá xuất sắc với vector TF-IDF siêu thưa (sparse) 3000 chiều như các model nền tảng tuyến tính/MLP.